# 04 - Evaluate, package, and compose

This notebook closes the loop: run direct PEFT inference, evaluate predictions, validate the adapter package, preview the Granite Switch Composer command, and optionally execute composition.

Keep LoRA and aLoRA composed checkpoints separate while testing parity.

In [ ]:
from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Repository root not found")


ROOT = find_repo_root(Path.cwd().resolve())
ADAPTER_NAME = "my_adapter"
TECHNOLOGY = "lora"
BASE_MODEL = "ibm-granite/granite-4.1-3b"
DATA_DIR = ROOT / "workspaces" / ADAPTER_NAME
ADAPTER_DIR = ROOT / "outputs" / "my-library" / ADAPTER_NAME / "granite-4.1-3b" / TECHNOLOGY
PREDICTIONS = DATA_DIR / f"predictions-{TECHNOLOGY}.jsonl"
COMPOSED_DIR = ROOT / "outputs" / "composed" / f"{ADAPTER_NAME}-{TECHNOLOGY}"
RUN_INFERENCE = False
RUN_COMPOSITION = False

## Direct PEFT inference

Generate predictions before composition. For aLoRA, add `--invocation` with the exact marker used in notebook 03.

In [ ]:
inference_command = [
    sys.executable,
    str(ROOT / "scripts" / "run_peft_adapter.py"),
    "--base-model",
    BASE_MODEL,
    "--adapter",
    str(ADAPTER_DIR),
    "--input-file",
    str(DATA_DIR / "test.jsonl"),
    "--output-file",
    str(PREDICTIONS),
]
if TECHNOLOGY == "alora":
    inference_command.extend(["--invocation", f"<{ADAPTER_NAME}>"])
print(" ".join(map(str, inference_command)))

if RUN_INFERENCE:
    subprocess.run(inference_command, cwd=ROOT, check=True)

## Evaluate predictions

This reports JSON validity, schema validity, exact match, missing outputs, and unexpected outputs.

In [ ]:
evaluation_command = [
    sys.executable,
    "-m",
    "granite_adapter_guide",
    "evaluate",
    str(DATA_DIR / "test.jsonl"),
    str(PREDICTIONS),
    "--schema",
    str(DATA_DIR / "schema.json"),
]
if PREDICTIONS.is_file():
    subprocess.run(evaluation_command, cwd=ROOT, check=True)
else:
    print("No prediction file yet; direct inference is still required")

## Package and validate

Copy the approved contract beside the final PEFT weights and validate the exact Composer input directory.

In [ ]:
if ADAPTER_DIR.is_dir() and (DATA_DIR / "io.yaml").is_file():
    shutil.copy2(DATA_DIR / "io.yaml", ADAPTER_DIR / "io.yaml")
    subprocess.run(
        [sys.executable, "-m", "granite_adapter_guide", "validate-adapter", str(ADAPTER_DIR)],
        cwd=ROOT,
        check=True,
    )
else:
    print("Adapter or io.yaml is missing; packaging skipped")

## Preview or execute composition

The default is a dry run that prints the exact Composer command. Enable execution only after installing the compose environment and accepting the direct-adapter quality result.

In [ ]:
compose_command = [
    sys.executable,
    str(ROOT / "scripts" / "compose_model.py"),
    "--base-model",
    BASE_MODEL,
    "--adapter",
    str(ADAPTER_DIR),
    "--output",
    str(COMPOSED_DIR),
]
if RUN_COMPOSITION:
    compose_command.append("--execute")
subprocess.run(compose_command, cwd=ROOT, check=True)

## Post-compose acceptance checklist

After composition:

- Read `compose_report.json` and reject missing or unexpected target modules.
- Confirm the adapter name in `config.json` and `adapter_index.json`.
- Compare direct PEFT and composed Hugging Face predictions on the same frozen test set.
- For aLoRA, verify the visible invocation marker and internal control-token boundary.
- Archive model, tokenizer, data, code, and adapter revisions with the final metrics.

vLLM serving is intentionally excluded from the Mac notebook path because the primary vLLM deployment path targets Linux/CUDA.